# Notebook zum QR-Algorithmus

### Vorbereitungen

Wir benötigten in diesem Notebook die folgenden Module:

In [1]:
import numpy as np
import scipy.linalg as spla  # für Matrixzerlegungen und co
import numpy.random as rnd   # für alles, was mit Zufallszahlen zu tun hat

Die Prozeduren, die wir in diesem Notebook betrachten, liefern verschiedene Vektoren und Matrizen als Ergebnis. Um diese schöner darstellen zu können, eignen sich die folgenden beiden Prozeduren. Was diese Prozeduren genau tun müssen Sie sich nicht anschauen.

In [2]:
def printvector(v):
    if v.dtype == "int":
        print(''.join([' {:4}'.format(item) for item in v])+"\n")
    elif v.dtype == "complex128":
        print(''.join([' {:16.3f}'.format(item) for item in v])+"\n")
    else:
        print(''.join([' {:7.3f}'.format(item) for item in v])+"\n")

In [3]:
def printmatrix(A):
    if A.dtype == "int":
        print('\n'.join([''.join(['  {:4}'.format(item) for item in row]) for row in A])+"\n")
    elif A.dtype == "complex128":
        print('\n'.join([''.join(['  {:16.3f}'.format(item) for item in row]) for row in A])+"\n")   
    else:
        print('\n'.join([''.join(['  {:7.3f}'.format(item) for item in row]) for row in A])+"\n")       

### Problemstellung & Modellmatrizen
In diesem Notebook wollen wir Eigenwerte von Matrizen mit verschiedenen Varianten des QR-Algorithmus approximieren. Zunächst konstruieren wir uns dazu ein paar Modellmatrizen, von denen wir die Eigenwerte kennen und mit denen wir die Algorithmen testen können.

Zur Konstruktion der Matrizen starten wir zunächst mit einer Diagonalmatrix oder einer Blockdiagonalmatrix $D$, und wenden dann eine beliebig ausgewählte Ähnlichkeitstransformation an, d.h. berechnen $A=S^{-1} D S$ für eine invertierbare Matrix $S$. Die Eigenwerte von $A$ entsprechen dann denen von $D$. Der Matrix $A$ selbst sieht man aber die Eigenwerte nicht direkt an.

Konkret verwenden wir
$$
D_1 = \begin{pmatrix} 2 \\ & 1 \\ && 5 \\ &&& -4 \\ &&&& \frac12 \end{pmatrix}, \qquad
D_2= \begin{pmatrix} 2+2\mathrm{i} \\ & 1 \\ && 5 \\ &&& -4+1\mathrm{i} \\ &&&& \frac12-3\mathrm{i} \end{pmatrix}, \qquad
D_3 = \begin{pmatrix} 2 \\ & 1 \\ && 5 \\ &&& 5 \\ &&&& \frac12 \end{pmatrix}, \qquad
D_4 = \begin{pmatrix} 2 \\ & 1 \\ && 5 \\ &&& -5 \\ &&&& \frac12 \end{pmatrix}, \qquad
$$
sowie
$$
D_5 = \begin{pmatrix} 2+2\mathrm{i} \\ & 1 \\ && 1 & -1 \\ && 1 & 1 \\ &&&& \frac12-3\mathrm{i} \end{pmatrix} 
\qquad \text{und} \qquad
D_6 = \begin{pmatrix} 2 \\ & 1 \\ && 1 & -1 \\ && 1 & 1 \\ &&&& \frac12 \end{pmatrix},
$$
und definieren dann $A_i = S^{-1} D_i S$ für $i=1,\ldots,6$ mit der invertierbaren Matrix
$$ 
S = \begin{pmatrix} 
     2 & -1 &  1 &  0 & -1 \\
    -1 &  1 &  2 &  2 &  2 \\
    -1 &  0 &  2 & -1 &  1 \\
    -1 &  2 &  2 &  2 &  1 \\
     2 & -1 &  2 &  0 & -1 
\end{pmatrix}.
$$ 

In [4]:
# (Block-)Diagonalmatrizen
A1 = np.diag(np.array([2, 1, 5, -4, 1/2]))
A2 = np.diag(np.array([2+2j, 1, 5, -4+1j, 1/2-3j]))
A3 = np.diag(np.array([2, 1, 5,  5, 1/2]))
A4 = np.diag(np.array([2, 1, 5, -5, 1/2]))
A5 = np.array([[2+2j,0,0,0,0],[0,1,0,0,0],[0,0,1,-1,0],[0,0,1,1,0],[0,0,0,0,1/2-3j]])
A6 = np.array([[2,0,0,0,0],[0,1,0,0,0],[0,0,1,-1,0],[0,0,1,1,0],[0,0,0,0,1/2]])

# Ähnlichkeitstransformation
S = np.array([
    [2, -1, 1, 0, -1],
    [-1, 1, 2, 2, 2],
    [-1, 0, 2, -1, 1],
    [-1, 2, 2, 2, 1],
    [2, -1, 2, 0, -1]
])
S_inv = spla.inv(S)

A1 = S_inv @ A1 @ S
A2 = S_inv @ A2 @ S
A3 = S_inv @ A3 @ S
A4 = S_inv @ A4 @ S
A5 = S_inv @ A5 @ S
A6 = S_inv @ A6 @ S

Für die Matrix $A_1$ gilt dann zum Beispiel:

In [5]:
print('A_1 =')
printmatrix(A1)

A_1 =
   16.000  -15.125   10.250  -11.500   -7.375
   15.000  -17.875    3.750  -16.500   -8.625
   -3.000    1.500   -1.000    0.000    1.500
   -7.000    9.250   -8.500   10.000    2.750
   10.000   -8.875   13.750   -6.500   -2.625



**(a) Nennen Sie kurz die "Herausforderungen" im Bezug auf die Eigenwerte von $D_3, \ldots, D_6$.**

- $A_3$ hat einen doppelten Eigenwert (wobei der dazugehörige Eigenraum Dimension 2 hat)
- $A_4$ hat zwei verschiedene dominierende Eigenwerte
- $A_5$ hat das komplex-konjugierte Eigenwertpaar $\lambda = 1 \pm 1 \mathrm{i}$, also insbesondere zwei betragsmäßig gleich große Eigenwerte
- $A_6$: Wie $A_5$, nur dass die Matrix $A_6$ selbst nur reelle Einträge hat

## 1.) Naiver QR-Algorithmus

**(b) Implementieren Sie eine Prozedur `qr_alg_naiv`, die den QR-Algorithmus in der naiven Version auf eine Matrix anwendet. Die Anzahl der Iterationen `kMax` soll dabei als Eingabeparameter übergeben werden.**

Berechnen Sie die QR-Zerlegungen mithilfe der in `scipy` enthalten Prozedur (also mit `spla.qr(...)`).

In [6]:
def qr_alg_naiv(A,kMax):
    for k in range(kMax):
        Q,R = spla.qr(A)
        A = R@Q
                
    return A

**(c) Wenden Sie die Prozedur zunächst auf die Matrizen $A_1$ und $A_2$ an. Wie viele Iterationen `kMax` brauchen Sie für gute Ergebnisse?**

In [28]:
A_res = qr_alg_naiv(A1,50)
print('Finale Marix:')
printmatrix(A_res)

Finale Marix:
    5.000   16.734   -9.269  -18.008  -38.105
    0.000   -4.000   -1.400   13.068    2.913
    0.000    0.000    2.000   -1.176   -7.512
    0.000   -0.000    0.000    1.000    0.042
   -0.000    0.000    0.000    0.000    0.500

Finale Marix:
    5.000   16.734   -9.269  -18.008   38.105
   -0.000   -4.000   -1.399   13.068   -2.913
    0.000   -0.000    2.000   -1.176    7.512
    0.000    0.000    0.000    1.000   -0.042
    0.000    0.000   -0.000   -0.000    0.500



In [8]:
A_res = qr_alg_naiv(A2,130)
print('Finale Marix:')
printmatrix(A_res)

Finale Marix:
      5.000+0.000j    -14.424+8.685j     21.052+3.839j   110.014-33.866j   -21.164-14.967j
     -0.000+0.000j     -4.000+1.000j     -0.928+0.989j      5.948+0.491j    -12.695-4.151j
      0.000+0.000j     -0.000+0.000j      0.500-3.000j    -25.025-5.873j     -0.656+4.710j
      0.000+0.000j      0.000+0.000j     -0.000-0.000j      2.000+2.000j      0.636-0.318j
     -0.000-0.000j     -0.000-0.000j      0.000+0.000j     -0.000-0.000j      1.000+0.000j



**(d) Wenden Sie die Prozedur nun jeweils mit `kMax = 50` auf die Matrizen $A_3,...,A_6$ an. Beobachten und erklären Sie die Ergebnisse.**

Matrix $A_3$ mit doppeltem Eigenwert: Algorithmus funktioniert problemlos.

In [9]:
A_res = qr_alg_naiv(A3,50)
print('Finale Marix:')
printmatrix(A_res)

Finale Marix:
    5.000    0.000  -13.017   26.367  -33.573
    0.000    5.000   -2.136   -4.420   -6.529
    0.000    0.000    2.000   -1.176   -7.512
   -0.000    0.000   -0.000    1.000    0.042
   -0.000   -0.000    0.000    0.000    0.500



Matrix $A_4$ mit zwei verschiedenen, betragsmäßig gleichgroßen Eigenwerten:

In [10]:
A_res = qr_alg_naiv(A4,50)
print('Finale Marix:')
printmatrix(A_res)

Finale Marix:
    0.654   19.832   -8.281  -25.582  -38.321
    1.239   -0.654   -3.486   10.392   -5.175
    0.000    0.000    2.000   -1.176   -7.512
    0.000   -0.000    0.000    1.000    0.042
   -0.000    0.000    0.000    0.000    0.500



Algorithmus kann die Eigenwertkomponenten zu den beiden betragsmäßig gleich großen Eigenwerten nicht trennen. Der Diagonalblock hat aber (zwangsweise) die richtigen Eigenwerte.

In [11]:
B_res = A_res[0:2,0:2]
lam = spla.eigvals(B_res)
print('Eigenwerte des 2x2-Blocks oben links')
printvector(lam)

Eigenwerte des 2x2-Blocks oben links
     5.000+0.000j    -5.000+0.000j



Matrix $A_5$ mit komplex konjugiertem Eigenwert-Paar: Ähnlich wie bei $A_4$.

In [12]:
A_res = qr_alg_naiv(A5,50)
print('Finale Marix:')
printmatrix(A_res)

Finale Marix:
      0.604-2.872j  -12.391-113.608j    11.652+28.680j      5.945-9.714j   -13.733+19.840j
     -0.005-0.004j      1.866+1.792j     -0.326-0.241j     -0.260+0.139j      0.542-0.525j
      0.000+0.001j      0.154-0.352j      1.598+0.080j     -0.902-0.623j      0.530+0.477j
     -0.000-0.000j      0.243+0.000j      0.963-0.628j      0.432-0.000j     -1.456+0.000j
      0.000+0.000j     -0.000-0.000j     -0.000+0.000j      0.000+0.000j      1.000-0.000j



In [13]:
B_res = A_res[2:4,2:4]
lam = spla.eigvals(B_res)
print('Eigenwerte des 2x2-Blocks')
printvector(lam)

Eigenwerte des 2x2-Blocks
     1.022+1.000j     1.008-0.920j



Matrix $A_6$ mit komplex konjugiertem Eigenwert-Paar: Ähnlich wie bei $A_4$.

In [14]:
A_res = qr_alg_naiv(A6,50)
print('Finale Marix:')
printmatrix(A_res)

Finale Marix:
    2.000    3.000   -3.397  -10.157  -35.332
   -0.000    1.549    1.117    0.816    0.354
    0.000   -1.165    0.451    1.478   -0.075
    0.000    0.000    0.000    1.000    0.042
    0.000   -0.000   -0.000    0.000    0.500



In [15]:
B_res = A_res[1:3,1:3]
lam = spla.eigvals(B_res)
print('Eigenwerte des 2x2-Blocks')
printvector(lam)

Eigenwerte des 2x2-Blocks
     1.000+1.000j     1.000-1.000j



## 2.) Transformation auf Hessenberg-Form

Als ersten Schritt hin zu einer effizienteren Implementierung wollen wir die Matrizen durch eine Ähnlichkeitstransformation auf Hessenberg-Form bringen.

**(e) Schreiben Sie eine Prozedur `hess`, die eine Matrix durch eine unitäre Ähnlichkeitstransformation auf Hessenberg-Form bringt (siehe Beweis von Satz 6.22).**

Hinweise:
* Berücksichtigen Sie, wie die richtige Householder-Transformation aussieht, die einen Vektor mit **komplexen** Einträgen auf ein Vielfaches des ersten Einheitsvektors spiegelt (siehe Skript).
* Die verwendeten Householder-Transformationen brauchen Sie nicht speichern, denn wir sind nur an den Eigenwerten von Matrizen interessiert, und die Hessenberg-Matrix, die Ihre Prozedur am Ende liefert, ist ja ähnlich zur Ausgangsmatrix.
* Bei Vektoren (genauer: eindimensionale `ndarray`) unterscheidet Numpy nicht zwischen Zeilen- und Spaltenvektoren, sondern interpretiert sie (zum Beispiel bei der Matrixmultilplikation) so wie es Sinn macht.
* Die komplex konjugierte Variante eines Vektors/arrays $v$ erhalten Sie über `v.conj()`.
* Das dyadische Produkt $vw^\top$ zweier Vektoren $v,w\in\mathbb{C}^n$ erhalten Sie über `np.outer(v,w)`.

In [16]:
def hess(A):
    n = np.size(A,0) # Anzahl Zeilen/Spalten
    for j in range(n-2):
        # Spiegele erste Spalte von A[j+1:,j:] auf alpha-faches von e_1
        
        # Baue passenden Householder-Vektor zusammen
        x = np.copy(A[j+1:,j]) # Erste Spalte von A[j+1:,j:]
        alpha = - x[0]/np.abs(x[0]) * np.linalg.norm(x)
        v = x
        v[0] -= alpha
        v = v/np.linalg.norm(v)
        
        # Wende Q = I - 2vv^H von links an
        A[j+1,j] = alpha
        A[j+2:,j] = 0
        A[j+1:,j+1:] += np.outer( (-2*v) , v.conj() @ A[j+1:,j+1:] )

        # Wende Q = I - 2vv^H von rechts an
        A[:,j+1:] -= np.outer( A[:,j+1:] @ (2*v) , v.conj() )

    return A

**(f) Wenden Sie Ihre Prozedur zum Test auf die Matrizen $A_1$ und $A_2$ an. Übergeben Sie dabei nur eine Kopie der Matrizen an die Prozedur `hess`, damit die Ausgangsmatrizen unverändert bleiben. Überprüfen Sie: Haben die Ergebnismatrizen Hessenberg-Struktur? Haben Sie weiterhin die selben Eigenwerte wie die Ausgangsmatrizen (über `spla.eigvals` überprüfbar)?**

In [17]:
print('Eigenwerte vorher:')
printvector(spla.eigvals(A1))

A1_hess = hess(A1.copy())
print('Matrix in Hessenbergform:')
printmatrix(A1_hess)

print('Eigenwerte nachher:')
printvector(spla.eigvals(A1_hess))

Eigenwerte vorher:
    -4.000+0.000j     5.000+0.000j     2.000+0.000j     0.500+0.000j     1.000+0.000j

Matrix in Hessenbergform:
   16.000   12.819  -11.423   14.699   -3.098
  -19.570  -16.389   16.033  -23.265    6.682
    0.000   -0.757    0.407   -3.797    6.616
    0.000    0.000   -4.424    3.517    8.044
    0.000    0.000    0.000    0.039    0.965

Eigenwerte nachher:
    -4.000+0.000j     5.000+0.000j     2.000+0.000j     0.500+0.000j     1.000+0.000j



In [18]:
print('Eigenwerte vorher:')
printvector(spla.eigvals(A2))

A2_hess = hess(A2.copy())
print('Matrix in Hessenbergform:')
printmatrix(A2_hess)

print('Eigenwerte nachher:')
printvector(spla.eigvals(A2_hess))

Eigenwerte vorher:
    -4.000+1.000j     0.500-3.000j     5.000-0.000j     2.000+2.000j     1.000+0.000j

Matrix in Hessenbergform:
    16.000+46.750j     8.848+35.339j    -0.451-20.130j    22.602+25.078j    -3.733+10.226j
   -23.534-61.580j   -13.510-46.700j     2.218+27.477j   -35.927-31.542j     5.042-14.034j
      0.000+0.000j     -0.026+0.353j     -3.756+1.070j     -2.716-0.912j     11.195+0.174j
      0.000+0.000j      0.000+0.000j     -3.012-0.302j      4.555-1.203j      5.529+0.976j
      0.000+0.000j      0.000+0.000j      0.000+0.000j     -0.445+0.415j      1.212+0.082j

Eigenwerte nachher:
    -4.000+1.000j     0.500-3.000j     5.000+0.000j     2.000+2.000j     1.000+0.000j



## 3.) QR-Algorithmus mit Shift für Hessenberg Matrizen

Bevor wir uns um den QR-Algorithmus mit Shift kümmern, bringen wir zunächst alle Modellmatrizen in Hessenberg-Form:

In [19]:
A1_hess = hess(A1.copy())
A2_hess = hess(A2.copy())
A3_hess = hess(A3.copy())
A4_hess = hess(A4.copy())
A5_hess = hess(A5.copy())
A6_hess = hess(A6.copy())

Nun wollen wir den QR-Algorithmus mit Shift für eine Hessenberg-Matrix $H$ implementieren. Dabei verwenden wird immer das letzte Element $\mu = h_{n,n}$ von $H$ als Shift. Wir iterieren so lange, bis Deflation auftritt, in dem Sinne, dass
$$
|h_{n,n-1}| \leq \texttt{eps} \left( |h_{n-1,n-1,}| + |h_{n,n}| \right)
$$
mit der Maschinengenauigkeit $\texttt{eps}$ gilt.

**(g) Implementieren Sie den eben beschriebenen QR-Algorithmus mit Shift. Sobald Deflation auftritt, soll die Prozedur beendet werden und dabei den isolierten Eigenwert sowie die Restmatrix ausgeben.**

Hinweise:
* Verwenden Sie wieder einen Parameter `kMax`, um die maximale Iterationszahl festzulegen, falls es nicht vorzeitig zu Deflation kommen sollte. In diesem Fall soll nur die letzte Iterierte ausgegeben werden.
* Die Maschinengenauigkeit (für 64Bit-Gleitkommazahlen) erhalten Sie über `np.finfo(np.float64).eps`.
* Stellen Sie sicher, dass Ihre Prozedur auch für $1\times1$-Matrizen ein sinnvolles Ergebnis liefert.

Testen Sie Ihre Prozedur mit den Matrizen `A1_hess` und `A2_hess`. Lassen Sie Python auch die Eigenwerte der Restmatrix berechnen und überprüfen Sie, ob die Ergebnisse Sinn ergeben.

In [20]:
def qr_alg_shift(H,kMax):
    ep = np.finfo(np.float64).eps # Maschinengenauigkeit
    n = np.size(H,0)
    if n == 1:
        print('Isolierter Eigenwert:',H)
        print('Keine Restmatrix')
        return
    else:
        for k in range(kMax):
            mu = H[-1,-1]
            Q,R = spla.qr(H-mu*np.eye(n))
            H = R@Q + mu*np.eye(n)

            if np.abs(H[-1,-2]) <= ep * (np.abs(H[-1,-1]) + np.abs(H[-2,-2])):
                print('Deflation in Schritt',k)
                print('Isolierter Eigenwert:',mu)
                print('Restmatrix:')
                printmatrix(H[0:-1,0:-1])
                print('Eigenwerte der Restmatrix:')
                printvector(spla.eigvals(H[0:-1,0:-1]))
                return 
                                
        print('Abbruch: kMax erreicht! Letzte Iterierte:')
        printmatrix(H)
        
    return []

In [21]:
printmatrix(A2_hess)

    16.000+46.750j     8.848+35.339j    -0.451-20.130j    22.602+25.078j    -3.733+10.226j
   -23.534-61.580j   -13.510-46.700j     2.218+27.477j   -35.927-31.542j     5.042-14.034j
      0.000+0.000j     -0.026+0.353j     -3.756+1.070j     -2.716-0.912j     11.195+0.174j
      0.000+0.000j      0.000+0.000j     -3.012-0.302j      4.555-1.203j      5.529+0.976j
      0.000+0.000j      0.000+0.000j      0.000+0.000j     -0.445+0.415j      1.212+0.082j



In [22]:
qr_alg_shift(A2_hess,100)

Deflation in Schritt 4
Isolierter Eigenwert: (1.000000000000004+4.128774029816907e-15j)
Restmatrix:
     -2.209+1.445j    -11.045-3.857j     9.396+16.481j    64.365-84.725j
      0.409+1.071j     -3.308-3.309j      0.184+3.165j     43.398-3.493j
      0.000+0.000j     -0.361+4.897j      7.799-0.278j   -12.630-27.281j
      0.000+0.000j      0.000+0.000j     -0.138-0.014j      1.218+2.143j

Eigenwerte der Restmatrix:
    -4.000+1.000j     0.500-3.000j     5.000+0.000j     2.000+2.000j



**(h) Erweitern Sie Ihre Prozedur aus Teil (g) folgendermaßen: Sobald Deflation auftritt, rufen Sie die Prozedur rekursiv auf, um den QR-Algorithmus mit der Restmatrix erneut zu starten. Speichern Sie dann den isolierten Eigenwert sowie die Eigenwerte der Restmatrix in einem Vektor, den Sie am Ende zurückgeben. Achten Sie hier besonders darauf, dass Ihre Prozedur für $1\times1$-Matrizen sinnvoll agiert.**

In [23]:
def qr_alg_shift(H,kMax):
    ep = np.finfo(np.float64).eps # Maschinengenauigkeit
    n = np.size(H,0)
    if n == 1:
        return H
    else:
        for k in range(kMax):
            mu = H[-1,-1]
            Q,R = spla.qr(H-mu*np.eye(n))
            H = R@Q + mu*np.eye(n)

            if np.abs(H[-1,-2]) <= ep * (np.abs(H[-1,-1]) + np.abs(H[-2,-2])):
                print('Deflation in Schritt',k,'. Isolierter EW ',mu,'. Größe Restmatrix',n-1,'x',n-1,'.')
                lam = qr_alg_shift(H[0:-1,0:-1],kMax)
                lam = np.append(lam,mu)
                return lam
                                
        print('Abbruch: kMax erreicht! Restmatrix:')
        printmatrix(H)
        return []

**(i) Sofern kein Abbruch wegen Erreichen der maximalen Iterationszahl erfolgt, sollte Ihre Prozedur aus Teil (h) einen Vektor mit allen Eigenwerten der eingegebenen Matrix $H$ berechnen. Überprüfen Sie dies zunächst anhand der Matrizen `A1_hess` und `A2_hess`. Wenn für diese Matrizen alles funktioniert, testen Sie auch die Matrizen `A3_hess`,...,`A6_hess`. Mit welchen Matrizen kommt die Prozedur (nicht) klar? Haben Sie eine Idee, warum? Beobachten Sie außerdem auch die Anzahl an Iterationen, die insgesamt durchgeführt wird, insbesondere im Vergleich zu den Aufgabenteilen (c) und (d).**

In [24]:
lam = qr_alg_shift(A4_hess,100)
print('Berechnete Eigenwerte:')
printvector(lam)

Deflation in Schritt 3 . Isolierter EW  1.0000000000013298 . Größe Restmatrix 4 x 4 .
Deflation in Schritt 3 . Isolierter EW  0.49999999989934174 . Größe Restmatrix 3 x 3 .
Deflation in Schritt 2 . Isolierter EW  1.9999999999999163 . Größe Restmatrix 2 x 2 .
Deflation in Schritt 2 . Isolierter EW  5.000000045821757 . Größe Restmatrix 1 x 1 .
Berechnete Eigenwerte:
  -5.000   5.000   2.000   0.500   1.000



In [25]:
lam = qr_alg_shift(A6_hess,100)
print('Berechnete Eigenwerte:')
printvector(lam)

Deflation in Schritt 3 . Isolierter EW  1.0000000000003704 . Größe Restmatrix 4 x 4 .
Deflation in Schritt 3 . Isolierter EW  0.4999999999999336 . Größe Restmatrix 3 x 3 .
Deflation in Schritt 25 . Isolierter EW  2.000000000231613 . Größe Restmatrix 2 x 2 .
Abbruch: kMax erreicht! Restmatrix:
    0.004    3.833
   -0.520    1.996

Berechnete Eigenwerte:
   2.000   0.500   1.000



Mit den Matrizen $A_1,...,A_5$ kommt der Algorithmus klar, nur mit $A_6$ nicht. Grund: $A_6$ ist eine reelle Matrix, und auch im Algorithmus bleibt dann alles reell. Das verursacht zwei Probleme:
* Die Einträge der Iterierten bleiben reell, also kann $h_{n,n}$ niemals einem der Eigenwerte $1\pm \mathrm{i}$ enstprechen.
* Die Shifts sind nur reell. Das heißt: Es gibt niemals einen Shift, der näher zum Eigenwert $1+1\mathrm{i}$ ist als zu $1-1\mathrm{i}$ oder anders herum. Der QR-Alg kann zwischen den beiden Eigenwerten "nicht unterscheiden".

Natürlich können wir unsere Prozedur jetzt auch auf andere Matrizen als die Modellmatrizen anwenden, und so die Eigenwerte (quasi) beliebiger Matrizen berechnen. Hier zum Beispiel für eine zufällige Matrix mit komplexen Einträgen: 

In [26]:
n = 20
A = rnd.rand(n,n) + 1j*rnd.rand(n,n)
A_hess = hess(A)

print('Eigenwerte laut Python:')
printvector(spla.eigvals(A_hess))

print('Eigenwerte selbst berechnet:')
printvector(qr_alg_shift(A_hess,50))

Eigenwerte laut Python:
   10.040+10.573j    -1.264+1.476j    -1.735-0.452j    -0.840+1.137j     0.547+1.217j    -0.867+0.716j     1.409-0.420j     0.959+0.635j     1.232+0.108j     0.986+0.356j    -0.335+0.731j    -1.099-0.745j    -0.324-1.560j     0.388-1.412j    -0.918-0.336j    -0.425+0.141j    -0.249-1.107j     0.024-0.955j     0.392-0.603j     0.215-0.140j

Eigenwerte selbst berechnet:
Deflation in Schritt 5 . Isolierter EW  (0.21527865135384994-0.13983384308539376j) . Größe Restmatrix 19 x 19 .
Deflation in Schritt 4 . Isolierter EW  (0.3923730453990151-0.6033818389361696j) . Größe Restmatrix 18 x 18 .
Deflation in Schritt 5 . Isolierter EW  (0.023947991785121397-0.9546983524882441j) . Größe Restmatrix 17 x 17 .
Deflation in Schritt 3 . Isolierter EW  (-0.24880464235559976-1.1074875979500738j) . Größe Restmatrix 16 x 16 .
Deflation in Schritt 2 . Isolierter EW  (-0.424539239752316+0.1411080745744003j) . Größe Restmatrix 15 x 15 .
Deflation in Schritt 3 . Isolierter EW  (-0.91755